# M2-02: Section Content Generation Testing

## Контекст

Этот notebook тестирует генерацию контента для **одного раздела документа** на основе утвержденной структуры.

### Отличия от M2-01

| Аспект | M2-01 (Planning) | M2-02 (Generation) |
|--------|------------------|-------------------|
| Цель | Структура документа | Контент раздела |
| Output | DocumentStructure | str (markdown text) |
| HITL | Да (двухфазная) | Нет |
| Complexity | FSM + Telegram bot | Простой LLM вызов |

### Workflow

```
Input:
  - Original topic (вопрос/тема)
  - Document structure (полная структура документа)
  - Current section (конкретный раздел для генерации)
  - External sources (конспекты, RAG, web search - опционально)

Process:
  LLM.invoke(system_prompt + context) → section content (markdown)

Output:
  - Markdown текст раздела (без заголовка раздела, с ### для подразделов)
```

### Цель тестирования

1. Проверить качество генерации раздела на основе структуры
2. Оценить интеграцию external sources (конспекты, будущие RAG/web)
3. Убедиться, что контент:
   - Следует структуре subsections → theses
   - Self-sufficient (все концепции объяснены)
   - Готов к сборке в document_assembly

## 1. Setup and Environment

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from jinja2 import Template

# Add project root to path для импорта из learnflow
project_root = Path().cwd().parent.parent
sys.path.insert(0, str(project_root))

# Загружаем переменные окружения
env_local = project_root / ".env.local"
env_file = project_root / ".env"

if env_local.exists():
    load_dotenv(env_local)
    print(f"✓ Loaded .env.local from {env_local}")
elif env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded .env from {env_file}")
else:
    print("⚠ No .env file found")

print(f"✓ Project root: {project_root}")

# Helper function to resolve API keys
def resolve_api_key(key_value: str) -> str:
    """
    Resolve API key from config value.
    If starts with $, read from environment variable.
    Otherwise, use the value directly.
    """
    if key_value.startswith("$"):
        env_var = key_value[1:]  # Remove $
        value = os.getenv(env_var)
        if not value:
            raise ValueError(f"Environment variable {env_var} not found")
        return value
    return key_value

# Helper function to create LLM with provider config
def create_llm(model_name: str, temperature: float, api_key: str, provider_config: dict = None):
    """
    Create ChatOpenAI instance with optional custom provider config.
    
    Args:
        model_name: Model name
        temperature: Temperature
        api_key: API key
        provider_config: Provider configuration dict (with base_url, etc.)
    
    Returns:
        ChatOpenAI instance
    """
    llm_kwargs = {
        "model": model_name,
        "temperature": temperature,
        "api_key": api_key
    }
    
    # Add base_url if configured
    if provider_config and provider_config.get("base_url"):
        llm_kwargs["base_url"] = provider_config["base_url"]
        print(f"  ✓ Using custom base_url: {provider_config['base_url']}")
    
    return ChatOpenAI(**llm_kwargs)

print("✓ Helper functions defined")

✓ Loaded .env.local from /home/bbaron/dev/my_pet_projects/learnflow-ai/.env.local
✓ Project root: /home/bbaron/dev/my_pet_projects/learnflow-ai
✓ Helper functions defined


## 2. Pydantic Models

Импортируем модели из `learnflow.models.document_structure`:

In [3]:
from learnflow.models.document_structure import (
    DocumentStructure,
    Section,
    Subsection,
)

print("✓ Models imported:")
print("  - DocumentStructure")
print("  - Section")
print("  - Subsection")

✓ Models imported:
  - DocumentStructure
  - Section
  - Subsection


## 3. Configuration

### Model Configuration

In [4]:
# LLM Configuration (will be loaded from M2-config.yaml)
print("⚠ MODEL_NAME and TEMPERATURE will be loaded from M2-config.yaml")

⚠ MODEL_NAME and TEMPERATURE will be loaded from M2-config.yaml


In [5]:
# ============================================================================
# LOAD CONFIGURATION AND INPUT DATA
# ============================================================================
import yaml
import json
from pathlib import Path

# Load M2 config
config_path = Path("M2-config.yaml")
if not config_path.exists():
    config_path = Path("../../M2-config.yaml")  # Try parent directory

if config_path.exists():
    with open(config_path) as f:
        m2_config = yaml.safe_load(f)
    print(f"✓ Loaded M2 configuration")
else:
    raise FileNotFoundError(f"M2-config.yaml not found")

# Resolve API keys from provider config
provider_config = m2_config.get("provider", {})
openai_api_key = resolve_api_key(provider_config.get("api_key", "$OPENAI_API_KEY"))

print(f"  Provider: {provider_config.get('name', 'openai')}")
print(f"  Base URL: {provider_config.get('base_url', 'default')}")
print(f"  API key loaded: {openai_api_key[:8]}...")

# Override model settings
MODEL_NAME = m2_config["model"]
TEMPERATURE = m2_config["temperature"]

print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")

# Load planning output (previous notebook)
outputs_dir = Path(m2_config["outputs_dir"])
planning_path = outputs_dir / m2_config["planning_output"]

if not planning_path.exists():
    raise FileNotFoundError(f"Planning output not found at {planning_path}")

with open(planning_path, encoding="utf-8") as f:
    planning_data = json.load(f)

# Reconstruct objects
topic = planning_data["topic"]
external_sources = planning_data["external_sources"]
document_structure = DocumentStructure(**planning_data["structure"])

# Assign order programmatically (LLM doesn't generate order field)
for section_idx, section in enumerate(document_structure.sections, 1):
    section.order = section_idx
    for subsection_idx, subsection in enumerate(section.subsections, 1):
        subsection.order = subsection_idx

print(f"\n✓ Loaded planning output from: {planning_path}")
print(f"  Topic: {topic[:60]}...")
print(f"  Sections: {len(document_structure.sections)}")
print(f"  External sources: {'Yes' if external_sources else 'No'}")

✓ Loaded M2 configuration
  Provider: openai
  Base URL: None
  API key loaded: sk-proj-...
  Model: gpt-4.1
  Temperature: 0.3

✓ Loaded planning output from: outputs/planning_output.json
  Topic: Schema-guided reasoning для разработчиков LLM-агентов...
  Sections: 3
  External sources: Yes


### Placeholder Configuration

Встроенная конфигурация для самодостаточности notebook (ML/intermediate уровень):

In [9]:
def build_placeholders(
    m2_config: dict,
    input_content: str,
    document_structure: DocumentStructure,
    current_section: Section,
    external_sources: str = ""
) -> dict:
    """
    Построение словаря placeholder значений из M2 конфигурации.
    
    Args:
        m2_config: Загруженная конфигурация из M2-config.yaml
        input_content: Оригинальная тема/вопрос
        document_structure: Полная структура документа (для контекста)
        current_section: Раздел для генерации
        external_sources: Внешние источники (конспекты, RAG, web search)
    
    Returns:
        dict со всеми placeholder значениями для Jinja2
    """
    ph_keys = m2_config["placeholders"]
    ph_values = m2_config["placeholder_values"]
    
    return {
        "subject_keywords": ph_values["subject_keywords"][ph_keys["subject_keywords_key"]],
        "role_perspective": ph_values["role_perspective"][ph_keys["role_perspective_key"]],
        "subject_name": ph_values["subject_name"][ph_keys["subject_name_key"]],
        "language": ph_values["language"][ph_keys["language_key"]],
        "target_audience_inline": ph_values["target_audience_inline"][ph_keys["target_audience_inline_key"]],
        "target_audience_block": ph_values["target_audience_block"][ph_keys["target_audience_block_key"]],
        "topic_coverage": ph_values["topic_coverage"][ph_keys["topic_coverage_key"]],
        "material_type_inline": ph_values["material_type_inline"][ph_keys["material_type_inline_key"]],
        "material_type_block": ph_values["material_type_block"][ph_keys["material_type_block_key"]],
        "explanation_depth": ph_values["explanation_depth"][ph_keys["explanation_depth_key"]],
        "style": ph_values["style"][ph_keys["style_key"]],
        # Context-specific
        "input_content": input_content,
        "document_structure": document_structure.model_dump_json(indent=2),
        "current_section": current_section.model_dump_json(indent=2),
        "external_sources": external_sources,
    }


print("✓ build_placeholders() function defined (reads from m2_config)")

✓ build_placeholders() function defined (reads from m2_config)


## 4. System Prompt

### Ключевые особенности:

1. **Гибкая интеграция источников** - `external_sources` может содержать:
   - Распознанные конспекты (handwritten notes)
   - RAG результаты (будущее)
   - Web search результаты (будущее)
   - Или быть пустым

2. **Scope control** - генерация ТОЛЬКО текущего раздела

3. **Assembly-ready** - выход готов к сборке в document_assembly:
   - Без заголовка раздела (добавится позже)
   - С `###` для subsections
   - Правильный порядок subsections

In [10]:
GENERATE_SECTION_SYSTEM_PROMPT = """
KEYWORD: {{ subject_keywords }}
<!-- Keywords above activate domain expertise, use naturally if relevant -->

<role>
You are a {{ role_perspective }} specializing in {{ subject_name }}, generating section content for {{ material_type_inline }} targeted at {{ target_audience_inline }}.
</role>

<task>
Generate comprehensive educational content for a SPECIFIC SECTION of a larger document based on the provided structure and context.
</task>

<input_data>
  <original_topic>
  {{ input_content }}
  </original_topic>

  <document_structure>
  {{ document_structure }}
  </document_structure>

  <current_section>
  {{ current_section }}
  </current_section>

  <external_sources>
  {{ external_sources }}
  </external_sources>
</input_data>

<generation_requirements>
  <scope>
    - Generate content ONLY for the current section specified in <current_section>
    - Follow the subsection structure and theses defined in current_section
    - Ensure each subsection thesis is thoroughly covered
    - Maintain awareness of overall document context from document_structure
    - Do not add content beyond the scope of current_section
  </scope>

  <source_integration>
    - When external_sources are provided, use them as primary reference material
    - Prioritize formulas, notations, methods, and explanations from external sources
    - External sources may include:
      * Handwritten student notes (recognized text from images)
      * RAG search results from knowledge bases
      * Web search results from current documentation
      * Any other contextual materials
    - Complement external sources with your domain expertise to ensure completeness
    - Fill knowledge gaps using your training when sources are incomplete
    - If no external sources provided, rely entirely on your domain expertise
    - Maintain consistency between external sources and generated content
  </source_integration>

  <content_parameters>
    <topic_coverage> {{ topic_coverage }} </topic_coverage>
    <explanation_depth> {{ explanation_depth }} </explanation_depth>
    <style> {{ style }} </style>
    <material_type> {{ material_type_block }} </material_type>
  </content_parameters>

  <mathematics>
    If mathematical formulas, concepts, or derivations are required for a complete understanding, present each formula with a **detailed, step-by-step derivation**.
    - Explain every symbol and each step of the logic.
    - After the mathematical exposition, show **how this formula or principle is used in practice**.
    - Illuminate the connection between mathematical theory and real-world application.
    - For inline mathematical formulas, use single dollar signs:
        Example: The speed of light: $c = 3 \times 10^8\ m/s$
    - For display (block) mathematical formulas, use double dollar signs:
        Example:
        $$
        \vec{F} = m \vec{a} = m \frac{d\vec{v}}{dt}
        $$
  </mathematics>

  <target_audience> {{ target_audience_block }} </target_audience>
</generation_requirements>

<output_requirements>
  <structure>
    - Start directly with subsection content (no section title - it will be added during assembly)
    - Use markdown subsection headers (### for subsections)
    - Follow the subsection order from current_section exactly
    - Ensure smooth flow and transitions between subsections
    - Each subsection must be clearly separated with a header
  </structure>

  <quality>
    - Self-sufficient content for this section - all concepts fully explained
    - Coherent narrative from start to finish within section scope
    - No forward references to content outside this section
    - No backward references assuming prior sections were read
    - Complete coverage of all theses from current_section
  </quality>

  <language> {{ language }} </language>
</output_requirements>

<output_format>
Generate section content directly without meta-commentary, introductory remarks, or process explanations.
Do NOT include the section title itself - only subsection content.
Start immediately with the first subsection using ### header.
</output_format>
"""

print("✓ GENERATE_SECTION_SYSTEM_PROMPT defined")
print(f"  Length: {len(GENERATE_SECTION_SYSTEM_PROMPT)} chars")

✓ GENERATE_SECTION_SYSTEM_PROMPT defined
  Length: 4092 chars


## 5. Generation Function

In [11]:
def generate_section_content(
    placeholders: dict,
    model_name: str = MODEL_NAME,
    temperature: float = TEMPERATURE,
    verbose: bool = True,
    provider_config: dict = None
) -> str:
    """
    Генерирует контент для одного раздела документа.
    
    Args:
        placeholders: Словарь со всеми placeholder значениями
        model_name: Название модели OpenAI
        temperature: Температура генерации (0.0-1.0)
        verbose: Выводить ли промежуточную информацию
        provider_config: Provider configuration (with base_url, etc.)
    
    Returns:
        str: Markdown контент раздела (без заголовка раздела)
    """
    # Создаем модель с provider config
    llm = create_llm(model_name, temperature, openai_api_key, provider_config)
    
    # Рендерим системный промпт через Jinja2
    template = Template(GENERATE_SECTION_SYSTEM_PROMPT)
    rendered_prompt = template.render(**placeholders)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Generating section content")
        print(f"Topic: {placeholders['input_content'][:60]}...")
        print(f"Model: {model_name} (temp={temperature})")
        has_sources = bool(placeholders.get('external_sources', '').strip())
        print(f"External sources: {'Yes' if has_sources else 'No'}")
        print(f"{'='*60}\n")
    
    # Генерируем
    messages = [SystemMessage(content=rendered_prompt)]
    response = llm.invoke(messages)
    content = response.content
    
    if verbose:
        print(f"✓ Generated {len(content)} characters")
        print(f"✓ Estimated tokens: ~{len(content) // 4}")
    
    return content


print("✓ generate_section_content() function defined")

✓ generate_section_content() function defined


## 6. Автоматическая генерация всех секций (M2 Flow)

In [12]:
# ============================================================================
# GENERATE ALL SECTIONS
# ============================================================================
from typing import List, Dict

# Storage for generated sections
generated_sections: List[Dict] = []

print(f"\n{'='*60}")
print("GENERATING SECTION CONTENT")
print(f"{'='*60}\n")

# Sequential generation (можно сделать параллельным позже)
for i, section in enumerate(document_structure.sections, 1):
    print(f"\n{'─'*60}")
    print(f"Section {i}/{len(document_structure.sections)}: {section.title}")
    print(f"{'─'*60}")
    
    # Build placeholders for this section from config
    placeholders = build_placeholders(
        m2_config=m2_config,
        input_content=topic,
        document_structure=document_structure,
        current_section=section,
        external_sources=external_sources
    )
    
    # Generate content with provider config
    content = generate_section_content(
        placeholders=placeholders,
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
        verbose=True,
        provider_config=provider_config
    )
    
    # Store with section metadata
    generated_sections.append({
        "order": section.order,
        "title": section.title,
        "content": content
    })
    
    print(f"✓ Section {i} completed\n")

# Sort by order (important for parallel generation)
generated_sections.sort(key=lambda x: x["order"])

print(f"\n{'='*60}")
print(f"✓ All sections generated: {len(generated_sections)}")
print(f"{'='*60}")


GENERATING SECTION CONTENT


────────────────────────────────────────────────────────────
Section 1/3: Введение в Schema-guided reasoning (SGR)
────────────────────────────────────────────────────────────

Generating section content
Topic: Schema-guided reasoning для разработчиков LLM-агентов...
Model: gpt-4.1 (temp=0.3)
External sources: Yes

✓ Generated 2224 characters
✓ Estimated tokens: ~556
✓ Section 1 completed


────────────────────────────────────────────────────────────
Section 2/3: Ключевые механизмы SGR
────────────────────────────────────────────────────────────

Generating section content
Topic: Schema-guided reasoning для разработчиков LLM-агентов...
Model: gpt-4.1 (temp=0.3)
External sources: Yes

✓ Generated 4256 characters
✓ Estimated tokens: ~1064
✓ Section 2 completed


────────────────────────────────────────────────────────────
Section 3/3: Практика и лучшие подходы
────────────────────────────────────────────────────────────

Generating section content
Topic: Sch

## 7. Document Assembly

In [13]:
# ============================================================================
# DOCUMENT ASSEMBLY
# ============================================================================

print(f"\n{'='*60}")
print("ASSEMBLING FINAL DOCUMENT")
print(f"{'='*60}\n")

# Build markdown document
markdown_parts = []

# Title (from topic)
markdown_parts.append(f"# {topic}\n\n")

# Sections
for section_data in generated_sections:
    # Section title (## for sections)
    markdown_parts.append(f"## {section_data['title']}\n\n")
    
    # Section content (уже содержит ### для subsections)
    markdown_parts.append(section_data['content'])
    markdown_parts.append("\n\n")

# Join all parts
final_document = "".join(markdown_parts)

print(f"✓ Document assembled")
print(f"  Total length: {len(final_document)} chars")
print(f"  Estimated tokens: ~{len(final_document) // 4}")


ASSEMBLING FINAL DOCUMENT

✓ Document assembled
  Total length: 10807 chars
  Estimated tokens: ~2701


## 8. Сохранение финального документа

In [14]:
# ============================================================================
# SAVE FINAL DOCUMENT
# ============================================================================
from pathlib import Path

# Save markdown document
outputs_dir = Path(m2_config["outputs_dir"])
output_path = outputs_dir / m2_config["final_document"]

with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_document)

print(f"\n{'='*60}")
print(f"✓ Final document saved to: {output_path}")
print(f"  Length: {len(final_document)} chars")
print(f"  Sections: {len(generated_sections)}")
print(f"{'='*60}")
print(f"\n🎉 M2 FLOW COMPLETED!")
print(f"\n📄 View document: {output_path}")

# Optional: Display preview
print(f"\n{'='*60}")
print("DOCUMENT PREVIEW (first 500 chars)")
print(f"{'='*60}\n")
print(final_document[:500] + "...")


✓ Final document saved to: outputs/final_document.md
  Length: 10807 chars
  Sections: 3

🎉 M2 FLOW COMPLETED!

📄 View document: outputs/final_document.md

DOCUMENT PREVIEW (first 500 chars)

# Schema-guided reasoning для разработчиков LLM-агентов

## Введение в Schema-guided reasoning (SGR)

### Суть и задачи SGR

Schema-guided reasoning (SGR) — это подход, при котором процесс рассуждения LLM-агента (Large Language Model agent) структурируется с помощью явных схем (schemas). В отличие от свободной генерации, SGR задаёт чёткие рамки: агент должен следовать заранее определённой структуре, что обеспечивает предсказуемость и контроль над его действиями.

**Структурирование reasoning LLM...


---

## 9. Test Cases (ЗАКОММЕНТИРОВАНО для M2 Flow)

**Примечание**: Оригинальные тесты закомментированы для автоматического прогона M2 flow.
Для запуска тестов раскомментируйте секции ниже.

In [ ]:
# ### Test Case 1: Simple Structure (ЗАКОММЕНТИРОВАНО)
# 
# # Test Case 1: Gradient Descent - простая структура
# # test1_topic = "Gradient Descent в машинном обучении"
# # test1_structure = DocumentStructure(...)
# # ...

print("⚠ Test cases commented out for M2 Flow")

### Test 2: Генерация с external sources (конспекты)

In [ ]:
# Build placeholders
placeholders_2 = build_placeholders(
    input_content=test2_topic,
    document_structure=test2_structure,
    current_section=test2_current_section,
    external_sources=test2_external_sources
)

# Generate
content_2 = generate_section_content(placeholders_2, verbose=True)

### Результат Test 2:

In [ ]:
print("\n" + "="*60)
print(f"Section: {test2_current_section.title}")
print("="*60 + "\n")

display(Markdown(content_2))

## 8. Quality Assessment

### Manual Review Checklist

Проверьте каждый сгенерированный раздел:

**Структурные требования:**
- [ ] Следует структуре subsections из current_section
- [ ] Все theses покрыты в соответствующих subsections
- [ ] Не включен заголовок раздела (только ### subsections)
- [ ] Использует `###` для заголовков subsections
- [ ] Subsections идут в правильном порядке

**Интеграция источников:**
- [ ] (Test 2) Использует формулы/код из external sources
- [ ] (Test 2) Приоритизирует notations из конспектов
- [ ] (Test 1) Генерирует качественный контент без источников
- [ ] Нет противоречий между источниками и сгенерированным текстом

**Качество контента:**
- [ ] Self-sufficient - все концепции объяснены
- [ ] Appropriate depth (intermediate level)
- [ ] Правильный style (balanced technical + accessible)
- [ ] Language: Russian с English technical terms
- [ ] Математические формулы используют `$` и `$$`
- [ ] Плавные переходы между subsections

**Соответствие scope:**
- [ ] Не выходит за рамки current_section
- [ ] Не ссылается на другие разделы документа
- [ ] Не предполагает чтение других разделов

### Сравнение: с источниками vs без источников

In [ ]:
print("Comparison Summary:")
print("="*60)
print(f"\nTest 1 (no sources):")
print(f"  Length: {len(content_1)} chars (~{len(content_1)//4} tokens)")
print(f"  Subsections expected: {len(test1_current_section.subsections)}")
print(f"  Subsections found: {content_1.count('###')}")

print(f"\nTest 2 (with sources):")
print(f"  Length: {len(content_2)} chars (~{len(content_2)//4} tokens)")
print(f"  Subsections expected: {len(test2_current_section.subsections)}")
print(f"  Subsections found: {content_2.count('###')}")
print(f"  Source integration: {'QdrantClient' in content_2}")
print(f"  Code blocks: {content_2.count('```')}")

print(f"\n{'='*60}")

## 9. Observations & Next Steps

### Наблюдения после тестирования

_(Заполните после выполнения тестов)_

**Что работает хорошо:**
- 

**Что требует улучшения:**
- 

**Идеи для промпта:**
- 

### Next Steps

1. **Если качество хорошее:**
   - [ ] Добавить промпт в `configs/prompts.yaml`
   - [ ] Добавить в Prompt Config Service database
   - [ ] Добавить node config в `configs/graph.yaml`
   - [ ] End-to-end тест через Telegram bot

2. **Если требуются улучшения:**
   - [ ] Итерировать промпт в этом notebook
   - [ ] Протестировать на дополнительных примерах
   - [ ] Уточнить инструкции по source_integration
   - [ ] Добавить примеры в few-shot learning (опционально)

3. **Подготовка к RAG/Web Search интеграции:**
   - [ ] Протестировать с различными типами external_sources
   - [ ] Убедиться в гибкости обработки пустых/частичных источников
   - [ ] Подготовить форматирование для RAG результатов